In [22]:
import requests
import pandas as pd
import time


In [23]:
API_KEY="8265bd1679663a7ea12ac168da84d2e8"

In [24]:
genre_url = f"https://api.themoviedb.org/3/genre/movie/list?api_key={API_KEY}&language=en-US"

genres = requests.get(genre_url).json()["genres"]

genre_dict = {}

for g in genres:
    genre_dict[g["id"]] = g["name"]

print(genre_dict)

{28: 'Action', 12: 'Adventure', 16: 'Animation', 35: 'Comedy', 80: 'Crime', 99: 'Documentary', 18: 'Drama', 10751: 'Family', 14: 'Fantasy', 36: 'History', 27: 'Horror', 10402: 'Music', 9648: 'Mystery', 10749: 'Romance', 878: 'Science Fiction', 10770: 'TV Movie', 53: 'Thriller', 10752: 'War', 37: 'Western'}


In [25]:
movies=[]

for page in range(1,501):
    url = f"https://api.themoviedb.org/3/movie/top_rated?api_key={API_KEY}&language=en-US&page={page}"

    data=requests.get(url).json()

    if "results" not in data:
        continue

    for movie in data["results"]:
        genre_names=[genre_dict[g] for g in movie["genre_ids"]]

        movies.append({
            "name":movie["title"],
            "description":movie["overview"],
            "genre":genre_names[0] if genre_names else "Unknown"
        })

print(page)

time.sleep(0.2)
#nut

500


In [26]:
df=pd.DataFrame(movies)

In [27]:
df.to_csv("tmdb_movies_dataset.csv",index=False,encoding="utf-8-sig")

In [28]:
df.head()

,name,description,genre
0,Accidental Partners,Two women discover they were both scammed by t...,Comedy
1,Swapped,"A small woodland creature and a majestic bird,...",Adventure
2,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,Drama
3,Michael,"The story of Michael Jackson, one of the most ...",Music
4,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",Drama


In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   name         10000 non-null  str  
 1   description  10000 non-null  str  
 2   genre        10000 non-null  str  
dtypes: str(3)
memory usage: 3.0 MB


In [30]:
df.isnull().sum()

name           0
description    0
genre          0
dtype: int64

In [31]:
df.genre.value_counts()

genre
Drama              2337
Comedy             1933
Action             1275
Horror              821
Animation           602
Adventure           532
Thriller            487
Crime               469
Romance             333
Science Fiction     308
Family              239
Fantasy             224
Mystery             123
War                  97
Western              75
Music                69
History              61
TV Movie             13
Unknown               2
Name: count, dtype: int64

In [32]:
df=df.dropna()

In [33]:
df=df.drop_duplicates()

In [34]:
df.head()

,name,description,genre
0,Accidental Partners,Two women discover they were both scammed by t...,Comedy
1,Swapped,"A small woodland creature and a majestic bird,...",Adventure
2,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,Drama
3,Michael,"The story of Michael Jackson, one of the most ...",Music
4,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",Drama


In [35]:
import nltk
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\XPS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\XPS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [42]:
import re 
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words=set(stopwords.words("english"))

lemmatizer=WordNetLemmatizer()

In [46]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)

    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

In [47]:
df["clean_description"] = df["description"].apply(clean_text)
df["clean_name"] = df["name"].apply(clean_text)

In [48]:
df["text"]=df["clean_name"]+" "+df["clean_description"]

In [49]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()

df['label']=encoder.fit_transform(df["genre"])

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer(max_features=5000)

X=tfidf.fit_transform(df["text"])
Y=df["label"]

In [54]:
from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test=train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [55]:
from sklearn.naive_bayes import MultinomialNB

model=MultinomialNB()
model.fit(X_train,Y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [56]:
from sklearn.metrics import accuracy_score

pred=model.predict(X_test)

print(accuracy_score(Y_test,pred))

0.3951975987993997


In [57]:
from sklearn.metrics import classification_report
print(classification_report(Y_test,pred))

              precision    recall  f1-score   support

           0       0.47      0.56      0.51       255
           1       0.57      0.04      0.07       106
           2       0.33      0.05      0.09       120
           3       0.41      0.60      0.48       386
           4       0.00      0.00      0.00        94
           5       0.35      0.79      0.48       467
           6       0.00      0.00      0.00        48
           7       0.00      0.00      0.00        45
           8       0.00      0.00      0.00        12
           9       0.80      0.24      0.37       164
          10       0.00      0.00      0.00        14
          11       0.00      0.00      0.00        25
          12       0.00      0.00      0.00        67
          13       0.00      0.00      0.00        62
          14       0.00      0.00      0.00         3
          15       0.00      0.00      0.00        97
          17       0.00      0.00      0.00        19
          18       0.00    

C:\Users\XPS\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\XPS\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\XPS\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [58]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(Y_test,pred)

In [59]:
df.head()

,name,description,genre,clean_description,clean_name,text,label
0,Accidental Partners,Two women discover they were both scammed by t...,Comedy,two woman discover scammed man also got pregna...,accidental partner,accidental partner two woman discover scammed ...,3
1,Swapped,"A small woodland creature and a majestic bird,...",Adventure,small woodland creature majestic bird two natu...,swapped,swapped small woodland creature majestic bird ...,1
2,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,Drama,imprisoned double murder wife lover upstanding...,shawshank redemption,shawshank redemption imprisoned double murder ...,5
3,Michael,"The story of Michael Jackson, one of the most ...",Music,story michael jackson one influential artist w...,michael,michael story michael jackson one influential ...,10
4,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",Drama,spanning year chronicle fictional italian amer...,godfather,godfather spanning year chronicle fictional it...,5


In [60]:
df.to_csv("preprocessing_tmdb_movies_dataset.csv",index=False,encoding="utf-8-sig")